# **Data Cleaning & Imputation**

## Objective
This notebook performs data cleaning and missing value imputation for both datasets:
- Dataset 1: E-Commerce dataset
- Dataset 2: Marketing Insights for E-Commerce Company datasets (Customers, Sales, Marketing, Discount, Tax)

## Output Files
- Cleaned_E_Commerce.csv
- Cleaned_Customers.csv
- Cleaned_Online_Sales.csv
- Cleaned_Marketing_Spend.csv
- Cleaned_Discount_Coupon.csv
- Cleaned_Tax_amount.csv

In [1]:
# setup
import os
import pandas as pd
from IPython.display import display

dataset1_path = '../data/raw/dataset1'
dataset2_path = '../data/raw/dataset2'
output_folder = '../data/cleaned_data'

os.makedirs(output_folder, exist_ok=True) # make sure the directory is already created

# clean dataset 1 first
files1 = [file for file in os.listdir(dataset1_path) if file.endswith((".csv", ".xlsx"))]

for file in files1:
    file_path = os.path.join(dataset1_path, file)

    if file.endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)

    print(f"FILE: {file} | Shape: {df.shape}") # FILE: filename | (rows, columns)
    # check missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0]


    if missing.empty:
        print("No missing values\n")
        output_file = file.replace(".xlsx", ".csv")
        output_path = os.path.join(output_folder, f"Cleaned_{output_file}")
        df.to_csv(output_path, index=False)
    else:
        percent = (missing / len(df)) * 100
        result = pd.DataFrame({
            "MissingCount": missing,
            "MissingPercent": percent
        }).sort_values("MissingPercent", ascending=False)

        display(result)

FILE: CustomersData.xlsx | Shape: (1468, 4)
No missing values

FILE: Discount_Coupon.csv | Shape: (204, 4)
No missing values

FILE: Marketing_Spend.csv | Shape: (365, 3)
No missing values

FILE: Online_Sales.csv | Shape: (52924, 10)
No missing values

FILE: Tax_amount.xlsx | Shape: (20, 2)
No missing values



## Data Quality Assessment

We examined dataset1 in the raw data folder to identify missing values and assess overall data quality.

### Summary of Findings

- **CustomersData.xlsx** (20 rows, 2 columns): No missing values.
- **Discount_Coupon.csv** (1468 rows, 4 columns): No missing values.
- **Marketing_Spend.csv** (21 rows, 4 columns): No missing values.
- **Online_Sales.csv** (365 rows, 3 columns): No missing values.
- **Tax_amount.xlsx** (52,924 rows, 10 columns): No missing values.

Overall, dataset 1 is clean and complete, with no missing data issues detected.

In [2]:
# dataset 2
df = pd.read_excel('../data/raw/E Commerce Dataset.xlsx', sheet_name='E Comm')


missing = df.isnull().sum()
missing = missing[missing > 0]

# check if E Commerce Dataset is cleaned or not
if missing.empty:
    print("No missing values")
else:
    percent = (missing / len(df)) * 100
    result = pd.DataFrame({
        "MissingCount": missing,
        "MissingPercent": percent
    }).sort_values("MissingPercent", ascending=False)
    display(result)

df_original = df.copy()
cols_with_missing_num = df[missing.index].select_dtypes(include=["int64", "float64"]).columns

,MissingCount,MissingPercent
DaySinceLastOrder,307,5.452931
OrderAmountHikeFromlastYear,265,4.706927
Tenure,264,4.689165
OrderCount,258,4.582593
CouponUsed,256,4.547069
HourSpendOnApp,255,4.529307
WarehouseToHome,251,4.458259


## Missing Value Imputation

To address the remaining missing values in the dataset, an imputation strategy was applied based on data types.

- Numerical features were imputed using the **median** value.
- Categorical features were imputed using the **mode** (most frequent value).

### Rationale

The median was chosen for numerical features as it is robust to outliers and provides a better central tendency for skewed distributions.

The mode was used for categorical features to preserve the most common category and maintain the distribution of the data.

### Implementation

Missing values were filled using the following approach:

- Numerical columns: replaced with median values.
- Categorical columns: replaced with the most frequent values.

Since the proportion of missing data is relatively small (less than 6%), imputation is preferred over removing rows to avoid unnecessary data loss.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# numeric
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# categorical
categorical_cols = df.select_dtypes(include=["object"]).columns
df[categorical_cols] = df[categorical_cols].fillna(df[categorical_cols].mode().iloc[0])

fig, axes = plt.subplots(len(cols_with_missing_num), 1,
                         figsize=(8, 4 * len(cols_with_missing_num)))

save_dir = "../assets/images/imputation"
os.makedirs(save_dir, exist_ok=True)
if len(cols_with_missing_num) == 1:
    axes = [axes]

for i, col in enumerate(cols_with_missing_num):
    ax = axes[i]

    sns.histplot(df_original[col].dropna(), color='blue',
                 label='Original', stat='density', kde=True, ax=ax, alpha=0.4)

    sns.histplot(df[col], color='red',
                 label='Imputed', stat='density', kde=True, ax=ax, alpha=0.4)

    ax.axvline(df_original[col].median(), color='blue', linestyle='--')
    ax.axvline(df[col].median(), color='red', linestyle='--')

    ax.set_title(f'Distribution of {col}')
    ax.legend()

    plt.savefig(f"{save_dir}/{col}.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

## Data insight

Median imputation preserves distribution shape across all numerical features,
with only minor local adjustments around the median.

This confirms that the imputation strategy is robust and does not introduce
significant bias into the dataset.

In [ ]:
input_path = '../data/cleaned_data'
files = [file for file in os.listdir(input_path)]

disguised_missing = {'', 'na', 'n/a', 'nan', 'null', 'none', 'unknown', '-', '?', '.', 'missing', ' '}

for f in files:
    file_path = os.path.join(input_path, f)
    print(f"=== Checking: {f} ===")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {f}: {e}")
        continue

    for col in df.columns:
        if df[col].dtype == 'object':
            unique_vals = df[col].dropna().unique()

            issues = []

            # Hidden whitespace
            has_whitespace = any(isinstance(x, str) and (x != x.strip()) for x in unique_vals)
            if has_whitespace:
                issues.append("Hidden whitespace")

            # Case inconsistency
            try:
                lower_vals = set(str(x).lower() for x in unique_vals)
                if len(lower_vals) < len(unique_vals):
                    issues.append("Case inconsistency")
            except:
                pass

            # Missing disguised
            has_disguised = any(str(x).strip().lower() in disguised_missing for x in unique_vals)
            if has_disguised:
                issues.append("Disguised missing ('NA', 'Unknown', etc.)")

            if issues or len(unique_vals) <= 20:
                print(f"Column '{col}':")
                if issues:
                    print(f"  - Detected Issues: {', '.join(issues)}")
                if len(unique_vals) <= 20:
                    print(f"  - Unique values ({len(unique_vals)}): {list(unique_vals)}")
                else:
                    print(f"  - Unique values count: {len(unique_vals)} (Too many to list, please inspect manually)")
                print()
    print("\n")

### Data Quality Insight

Some categorical variables contain inconsistent representations of the same concept.

- *PreferredLoginDevice*: "Mobile Phone" and "Phone"
- *PreferredPaymentMode*: "CC" vs "Credit Card", "COD" vs "Cash on Delivery", "E wallet"
- *PreferedOrderCat*: "Mobile" and "Mobile Phone"

These inconsistencies may lead to incorrect analysis and unreliable model results.

### Next Steps

The categories will be standardized by merging equivalent values and applying consistent naming.

This ensures clean and reliable features for downstream tasks such as clustering.

In [ ]:
df = pd.read_csv('../data/cleaned_data/Cleaned_E_Commerce_Dataset.csv')
# PreferredLoginDevice
df['PreferredLoginDevice'] = df['PreferredLoginDevice'].replace({
    'Phone': 'Mobile Phone'
})

# PreferredPaymentMode
df['PreferredPaymentMode'] = df['PreferredPaymentMode'].replace({
    'CC': 'Credit Card',
    'COD': 'Cash on Delivery',
    'E wallet': 'E-Wallet'
})

# PreferedOrderCat
df['PreferedOrderCat'] = df['PreferedOrderCat'].replace({
    'Mobile': 'Mobile Phone'
})

In [ ]:
missing = df.isnull().sum().sum()

# check if E Commerce Dataset is cleaned or not. Save the cleaned data in /data/cleaned_data
if missing == 0:
    output_path = os.path.join(output_folder, f"Cleaned_E_Commerce_Dataset.csv")
    df.to_csv(output_path, index=False)
    print("No missing values")